### Descriptive Analysis


# Phân tích mô tả

Phần này nhằm tái hiện bức tranh tổng thể về hoạt động kinh doanh dựa trên dữ liệu lịch sử từ năm 2012 đến 2022.  

Mục tiêu là hiểu rõ xu hướng doanh thu, hiệu suất sản phẩm, đặc điểm khách hàng và hiệu quả vận hành của doanh nghiệp.

## Thiết lập thư viện và môi trường

Trong phần này, chúng tôi import các thư viện Python cần thiết phục vụ cho việc xử lý dữ liệu, trực quan hóa và phân tích thống kê.

- **pandas, numpy**: dùng để xử lý dữ liệu và tính toán số học  
- **matplotlib, seaborn**: dùng để trực quan hóa dữ liệu  
- **os, re**: dùng để thao tác với file và xử lý chuỗi  
- **scipy**: dùng cho các phép phân tích thống kê  

Các thư viện này là nền tảng cho quá trình phân tích dữ liệu khám phá (Descriptive Analysis) ở các phần tiếp theo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re as r
from scipy import stats

: 

## Nạp dữ liệu (tự động nhiều file)

Trong bước này, chúng tôi tiến hành nạp toàn bộ các file CSV từ thư mục dữ liệu vào môi trường làm việc.

Mỗi file sẽ được tự động đọc và gán vào một DataFrame tương ứng theo quy tắc đặt tên `df_<tên_file>`.  
Cách làm này giúp xử lý nhiều bộ dữ liệu một cách hiệu quả mà không cần phải đọc từng file thủ công.

Việc thiết lập này đảm bảo tất cả các nguồn dữ liệu cần thiết đã sẵn sàng cho các bước phân tích tiếp theo.

In [ ]:
path = r'C:\Users\ACER\Documents\datathon-2026-round-1\Datathon_Contest_UITUT\Data' 

# 2. Lấy danh sách tất cả các file trong thư mục
files = [f for f in os.listdir(path) if f.endswith('.csv')]

for file in files:
    # Tách tên file và phần mở rộng (vd: 'data.csv' -> 'data')
    file_name = os.path.splitext(file)[0]
    
    # Tạo đường dẫn đầy đủ
    full_path = os.path.join(path, file)
    
    # Đọc file và gán vào biến có tên df_tên_file
    # globals() giúp tạo biến động trong môi trường hiện tại
    globals()[f"df_{file_name}"] = pd.read_csv(full_path)
    
    print(f"Đã load xong: df_{file_name}")

## Tổng quan cấu trúc dữ liệu

Trong bước này, chúng tôi tiến hành khám phá cấu trúc của tất cả các bộ dữ liệu đã được nạp vào hệ thống.

Đối với mỗi bảng, chúng tôi xem xét:
- Danh sách các cột  
- Số lượng biến (columns)  

Bước này giúp cung cấp cái nhìn tổng quan về dữ liệu hiện có, đồng thời xác định các thuộc tính quan trọng phục vụ cho các bước phân tích tiếp theo.

In [ ]:
# Giả sử files là danh sách tên file đã lấy ở bước trước
count = 0
for file in files:
    file_name = os.path.splitext(file)[0]
    var_name = f"df_{file_name}"
    count = count + 1
    # Kiểm tra xem biến đó có tồn tại trong globals không để tránh lỗi
    if var_name in globals():
        df = globals()[var_name]
        print(f"--- Bảng: {var_name} ---")
        print(f"Danh sách cột: {list(df.columns)}")
        print(f"Số lượng cột: {len(df.columns)}")
        print("-" * 30)
print("SỐ LƯỢNG BẢNG: ", count)

## Đánh giá chất lượng dữ liệu

Trong bước này, chúng tôi tiến hành đánh giá tổng thể chất lượng của tất cả các bộ dữ liệu bằng cách xây dựng một bảng tổng hợp (data health report).

Đối với mỗi bảng dữ liệu, chúng tôi tính toán các chỉ số quan trọng bao gồm:
- Tổng số dòng và số cột  
- Số lượng giá trị thiếu (null)  
- Số lượng bản ghi trùng lặp  
- Tỷ lệ dữ liệu bị thiếu (%)  
- Dung lượng bộ nhớ sử dụng  

Việc đánh giá này giúp phát hiện các vấn đề tiềm ẩn như dữ liệu thiếu, trùng lặp hoặc không nhất quán, từ đó ảnh hưởng đến độ tin cậy của quá trình phân tích.

Kết quả thu được cung cấp cái nhìn tổng quan về chất lượng dữ liệu và là cơ sở để thực hiện các bước làm sạch dữ liệu tiếp theo.

In [ ]:
import pandas as pd

def create_data_health_report():
    # 1. Lấy danh sách các DataFrame đang có trong bộ nhớ (biến bắt đầu bằng df_)
    all_dfs = {k: v for k, v in globals().items() if k.startswith('df_') and isinstance(v, pd.DataFrame)}
    
    # Loại bỏ chính cái bảng report nếu nó đã tồn tại để tránh đệ quy
    all_dfs = {k: v for k, v in all_dfs.items() if k != 'df_final_report'}

    report_data = []

    for name, df in all_dfs.items():
        # Tính toán các thông số cơ bản
        total_rows = len(df)
        total_cols = len(df.columns)
        null_count = df.isnull().sum().sum()
        duplicate_count = df.duplicated().sum()

        # Tính % dữ liệu bị thiếu
        total_cells = total_rows * total_cols
        missing_rate = (null_count / total_cells) * 100 if total_cells > 0 else 0

        # Xử lý Memory Usage cho dễ đọc (đổi sang MB nếu > 1024 KB)
        mem_bytes = df.memory_usage(deep=True).sum()
        if mem_bytes < 1024 * 1024:
            mem_display = f"{mem_bytes / 1024:.2f} KB"
        else:
            mem_display = f"{mem_bytes / (1024 * 1024):.2f} MB"

        # Lưu vào list
        report_data.append({
            'Table Name': name,
            'Total Rows': total_rows,
            'Total Cols': total_cols,
            'Total Null/NaN': null_count,
            'Duplicates': duplicate_count,
            'Missing Rate (%)': round(missing_rate, 2),
            'Memory Usage': mem_display
        })

    # 2. Tạo DataFrame từ list kết quả và sắp xếp theo bảng có nhiều Null nhất hoặc nặng nhất
    health_report = pd.DataFrame(report_data)
    
    if not health_report.empty:
        health_report = health_report.sort_values(by='Total Null/NaN', ascending=False)
        
    return health_report

# 3. Chạy và hiển thị bảng báo cáo
df_final_report = create_data_health_report()

print("\n🚀 --- BẢNG TỔNG HỢP CHẤT LƯỢNG DỮ LIỆU --- 🚀")
# Kiểm tra nếu máy có cài thư viện tabulate để dùng to_markdown
try:
    print(df_final_report.to_markdown(index=False))
except ImportError:
    print(df_final_report.to_string(index=False))


## Phân tích doanh thu và giá vốn

**Mục tiêu:**  
Đánh giá sự biến động của doanh thu (Revenue) và giá vốn hàng bán (COGS) theo thời gian ở cấp độ năm.

**Ý nghĩa:**  
Phân tích này giúp nhận diện xu hướng tăng trưởng của doanh nghiệp, đồng thời đánh giá mối quan hệ giữa doanh thu và chi phí nhằm hiểu rõ hơn về hiệu quả kinh doanh.

**Phương pháp:**  
Dữ liệu được tổng hợp theo từng năm dựa trên ngày giao dịch, sau đó tính tổng doanh thu và giá vốn tương ứng.  
Biểu đồ đường được sử dụng để trực quan hóa xu hướng và so sánh sự thay đổi qua các năm.

### Nhận định chính

- Doanh thu có xu hướng tăng/giảm theo thời gian, phản ánh sự thay đổi trong hiệu suất kinh doanh  
- Giá vốn (COGS) biến động cùng chiều với doanh thu, cho thấy chi phí sản xuất/bán hàng tỷ lệ thuận với doanh thu  
- Khoảng cách giữa doanh thu và giá vốn thể hiện xu hướng lợi nhuận gộp, là chỉ số quan trọng đánh giá hiệu quả hoạt động  
- Những biến động lớn theo từng giai đoạn có thể xuất phát từ yếu tố mùa vụ, chiến dịch marketing hoặc thay đổi trong chiến lược kinh doanh

In [ ]:
df_sales['Date'] = pd.to_datetime(df_sales['Date'])

# Tạo cột Year và tính tổng theo năm
df_yearly = df_sales.groupby(df_sales['Date'].dt.year).agg({
    'Revenue': 'sum',
    'COGS': 'sum'
}).reset_index()

# 2. Vẽ biểu đồ
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Vẽ đường Revenue & COGS theo Năm
plt.plot(df_yearly['Date'], df_yearly['Revenue'], marker='o', label='Revenue', color='#2ecc71', linewidth=3)
plt.plot(df_yearly['Date'], df_yearly['COGS'], marker='s', label='COGS', color='#e74c3c', linewidth=3)

# 3. Trang trí
plt.title('BIỂU ĐỒ DOANH THU & GIÁ VỐN THEO NĂM', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Năm', fontsize=12)
plt.ylabel('Giá trị', fontsize=12)

# Hiển thị số liệu (Data Labels) theo đơn vị tỷ (hoặc triệu tùy ông)
for i in range(len(df_yearly)):
    plt.text(df_yearly['Date'][i], df_yearly['Revenue'][i], f"{df_yearly['Revenue'][i]/1e6:.1f}M", 
             va='bottom', ha='center', fontsize=10, color='#27ae60', fontweight='bold')

# Fix trục X chỉ hiển thị số năm nguyên (2012, 2013...)
plt.xticks(df_yearly['Date'].astype(int)) 

plt.legend()
plt.tight_layout()
plt.show()

## Phân tích cơ cấu sản phẩm — Tỷ suất lợi nhuận theo phân khúc

**Mục tiêu:**  
Xác định các phân khúc sản phẩm có tỷ suất lợi nhuận gộp cao nhất.

**Ý nghĩa:**  
Phân tích này giúp nhận diện những nhóm sản phẩm mang lại lợi nhuận cao, từ đó hỗ trợ doanh nghiệp tối ưu chiến lược kinh doanh và phân bổ nguồn lực hiệu quả.

**Phương pháp:**  
Tỷ suất lợi nhuận gộp được tính theo công thức: *(Giá bán - Giá vốn) / Giá bán*.  
Sau đó, dữ liệu được gom nhóm theo phân khúc (segment), tính giá trị trung bình và chọn ra top 8 phân khúc có tỷ suất lợi nhuận cao nhất.  
Kết quả được trực quan hóa bằng biểu đồ cột ngang.

### Nhận định chính

- Một số phân khúc sản phẩm có tỷ suất lợi nhuận gộp cao hơn đáng kể so với phần còn lại  
- Các phân khúc này đóng vai trò quan trọng trong việc tối đa hóa lợi nhuận tổng thể của doanh nghiệp  
- Sự khác biệt về tỷ suất lợi nhuận cho thấy sự chênh lệch về chi phí, chiến lược giá hoặc nhu cầu thị trường  
- Doanh nghiệp nên ưu tiên phát triển và đẩy mạnh các phân khúc có lợi nhuận cao
- Có thể tập trung marketing và tối ưu tồn kho cho các phân khúc có tỷ suất lợi nhuận cao nhằm gia tăng hiệu quả kinh doanh

In [ ]:

print("TOP 8 SEGMENTS CÓ TỶ SUẤT LỢI NHUẬN GỘP CAO NHẤT")
# 1. Tính Tỷ suất lợi nhuận gộp cho từng sản phẩm
# Công thức: (Price - COGS) / Price
df_products['gross_margin'] = (df_products['price'] - df_products['cogs']) / df_products['price']

# 2. Gom nhóm theo Segment và tính trung bình (Mean)
# Sau đó sắp xếp giảm dần và lấy 8 thằng đầu bảng
df_top8_segment = df_products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False).head(8).reset_index()

# Chuyển về % cho dễ đọc
df_top8_segment['margin_pct'] = df_top8_segment['gross_margin'] * 100

# 3. Vẽ biểu đồ cột ngang (Horizontal Bar Chart)
plt.figure(figsize=(12, 8))
colors = plt.cm.viridis(range(0, 256, 32)) # Tạo bảng màu cho đẹp

bars = plt.barh(df_top8_segment['segment'], df_top8_segment['margin_pct'], color=colors)

# 4. Trang trí biểu đồ
plt.title('TOP 8 SEGMENTS CÓ TỶ SUẤT LỢI NHUẬN GỘP CAO NHẤT', fontsize=15, fontweight='bold', pad=20)
plt.xlabel('Tỷ suất lợi nhuận (%)', fontsize=12)
plt.ylabel('Phân khúc (Segment)', fontsize=12)
plt.gca().invert_yaxis()  # Đảo ngược trục Y để thằng cao nhất nằm trên cùng
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Ghi số liệu % trực tiếp lên đầu cột
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.5, bar.get_y() + bar.get_height()/2, 
             f'{width:.2f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Phân tích cơ cấu sản phẩm — Doanh thu theo sản phẩm

**Mục tiêu:**  
Xác định các sản phẩm có tổng doanh thu cao nhất.

**Ý nghĩa:**  
Phân tích này giúp nhận diện các sản phẩm chủ lực đóng góp lớn vào doanh thu, từ đó hỗ trợ doanh nghiệp tối ưu chiến lược kinh doanh và danh mục sản phẩm.

**Phương pháp:**  
Dữ liệu được kết hợp từ các bảng đơn hàng, chi tiết đơn hàng và sản phẩm để xây dựng một bảng dữ liệu tổng hợp.  
Doanh thu thực tế được tính bằng công thức: *Giá bán thực tế × Số lượng*.  
Sau đó, dữ liệu được gom nhóm theo tên sản phẩm và tính tổng doanh thu, lựa chọn top 8 sản phẩm có doanh thu cao nhất.  
Kết quả được trực quan hóa bằng biểu đồ cột ngang.

In [ ]:
df_master = df_order_items.merge(df_orders[['order_id', 'order_date']], on='order_id', how='left')

# 2. Nối tiếp với df_products để lấy tên sản phẩm (product_name)
df_master = df_master.merge(df_products[['product_id', 'product_name']], on='product_id', how='left')

# 3. Tính Doanh thu thực tế theo từng dòng
# Công thức: unit_price (giá bán thực tế) * quantity (số lượng)
df_master['actual_revenue'] = df_master['unit_price'] * df_master['quantity']

# 4. Aggregate: Tính tổng doanh thu theo tên sản phẩm và lấy Top 8
df_top8_revenue = df_master.groupby('product_name')['actual_revenue'].sum().sort_values(ascending=False).head(8).reset_index()

# 5. Vẽ biểu đồ cột ngang
plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")

# Vẽ biểu đồ với tone màu 'rocket' nhìn cho nó rực cháy
chart = sns.barplot(
    data=df_top8_revenue, 
    x='actual_revenue', 
    y='product_name', 
    palette='rocket'
)

# 6. Trang trí
plt.title('📊 TOP 8 SẢN PHẨM CÓ TỔNG DOANH THU THỰC TẾ CAO NHẤT', fontsize=15, fontweight='bold', pad=20)
plt.xlabel('Tổng doanh thu (Unit Price * Quantity)', fontsize=12)
plt.ylabel('Tên sản phẩm', fontsize=12)

# Thêm nhãn số tiền lên đầu cột, định dạng có dấu phẩy ngăn cách nghìn
for container in chart.containers:
    chart.bar_label(container, fmt='{:,.0f}', padding=5, fontweight='bold')

plt.tight_layout()
plt.show()

## Phân tích vòng đời sản phẩm — Biến động doanh thu theo thời gian

**Mục tiêu:**  
Phân tích sự biến động doanh thu theo thời gian của các sản phẩm có doanh thu cao nhất.

**Ý nghĩa:**  
Giúp hiểu rõ xu hướng phát triển, vòng đời và mức độ ổn định doanh thu của từng sản phẩm theo thời gian.  
Từ đó có thể nhận diện các sản phẩm đang tăng trưởng, bão hòa hoặc suy giảm.

**Phương pháp:**  
Lựa chọn top 8 sản phẩm có tổng doanh thu cao nhất, sau đó tổng hợp doanh thu theo từng tháng.  
Dữ liệu được trực quan hóa bằng biểu đồ đường để theo dõi xu hướng doanh thu của từng sản phẩm theo thời gian.

### Nhận định chính

- Doanh thu của các sản phẩm có xu hướng biến động theo thời gian, phản ánh vòng đời sản phẩm  
- Một số sản phẩm duy trì doanh thu ổn định trong thời gian dài, cho thấy nhu cầu bền vững  
- Một số sản phẩm có xu hướng tăng trưởng mạnh hoặc suy giảm theo từng giai đoạn  
- Sự khác biệt về xu hướng giữa các sản phẩm cho thấy mức độ cạnh tranh và thay đổi trong nhu cầu thị trường
- Doanh nghiệp có thể tận dụng các sản phẩm đang tăng trưởng để mở rộng kinh doanh, đồng thời xem xét chiến lược cải tiến hoặc thay thế đối với các sản phẩm có xu hướng suy giảm

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# 1. Chuẩn bị dữ liệu
df_master['order_date'] = pd.to_datetime(df_master['order_date'])
df_master['actual_revenue'] = df_master['unit_price'] * df_master['quantity']

# 2. Lấy danh sách Top 8 (vẫn dựa trên tổng doanh thu lịch sử)
top_8_names = df_master.groupby('product_name')['actual_revenue'].sum().sort_values(ascending=False).head(8).index

# 3. Lọc lấy 8 ông nội này
df_top8_ts = df_master[df_master['product_name'].isin(top_8_names)].copy()

# 4. Gom nhóm theo THÁNG (Month)
# Dùng Grouper để dữ liệu liên tục theo dòng thời gian
df_plot_ts = df_top8_ts.groupby([pd.Grouper(key='order_date', freq='MS'), 'product_name'])['actual_revenue'].sum().reset_index()

# 5. Vẽ biểu đồ
plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

ax = sns.lineplot(
    data=df_plot_ts, 
    x='order_date', 
    y='actual_revenue', 
    hue='product_name', 
    marker=None, # Bỏ marker đi cho đỡ rối vì dữ liệu theo tháng rất dày
    linewidth=2
)

# 6. CHIÊU THỨC: Chỉ hiển thị NĂM trên trục X
ax.xaxis.set_major_locator(mdates.YearLocator()) # Chỉ đánh mốc tại các năm
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y')) # Định dạng hiển thị là số Năm (2012, 2013...)

# 7. Trang trí
plt.title('CHU KỲ SỐNG SẢN PHẨM - BIẾN ĐỘNG DOANH THU THÁNG CỦA TOP 8 SẢN PHẨM (Đơn vịr)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Thời gian (Năm)', fontsize=12)
plt.ylabel('Doanh thu tháng (Actual Revenue)', fontsize=12)

plt.legend(title='Sản phẩm', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Phân tích doanh thu theo năm — So sánh hiệu suất sản phẩm

**Mục tiêu:**  
So sánh doanh thu của các sản phẩm theo từng năm để đánh giá hiệu suất theo thời gian.

**Ý nghĩa:**  
Giúp xác định sản phẩm nào duy trì doanh thu ổn định, sản phẩm nào tăng trưởng hoặc suy giảm qua các năm.  
Đồng thời hỗ trợ nhận diện sản phẩm chủ lực trong dài hạn.

**Phương pháp:**  
Dữ liệu được tổng hợp theo năm và chuyển thành dạng bảng pivot với:
- Hàng: Tên sản phẩm  
- Cột: Năm  
- Giá trị: Tổng doanh thu  

Ngoài ra, tính thêm tổng doanh thu qua tất cả các năm để xác định sản phẩm có đóng góp lớn nhất.

### Nhận định chính

- Một số sản phẩm duy trì doanh thu ổn định qua nhiều năm, cho thấy nhu cầu bền vững  
- Một số sản phẩm có xu hướng tăng trưởng rõ rệt theo thời gian  
- Một số sản phẩm có dấu hiệu suy giảm, có thể đã bước vào giai đoạn bão hòa hoặc lỗi thời  
- Sản phẩm có tổng doanh thu cao nhất qua nhiều năm có thể được xem là sản phẩm chủ lực của doanh nghiệp
- Doanh nghiệp nên tập trung phát triển các sản phẩm có xu hướng tăng trưởng ổn định, đồng thời xem xét cải tiến hoặc thay thế các sản phẩm có dấu hiệu suy giảm

In [ ]:
# 1. Tạo thêm cột Năm để dễ pivot
df_top8_ts['year'] = df_top8_ts['order_date'].dt.year

# 2. Tạo bảng Pivot: Sản phẩm làm index, Năm làm cột, Doanh thu làm giá trị
df_pivot_table = df_top8_ts.pivot_table(
    index='product_name', 
    columns='year', 
    values='actual_revenue', 
    aggfunc='sum',
    fill_value=0  # Nếu năm đó sản phẩm không bán được thì để là 0
)

# 3. Tính thêm cột "Tổng cộng" để xem ông nào là 'vua doanh thu' thực sự
df_pivot_table['Total_All_Years'] = df_pivot_table.sum(axis=1)

# 4. Sắp xếp lại theo tổng doanh thu từ cao xuống thấp
df_pivot_table = df_pivot_table.sort_values(by='Total_All_Years', ascending=False)

# 5. Format số liệu cho dễ nhìn (thêm dấu phẩy ngăn cách hàng nghìn)
pd.options.display.float_format = '{:,.0f}'.format

print("--- BẢNG THEO DÕI DOANH THU TOP 8 SẢN PHẨM QUA CÁC NĂM ---")
print(df_pivot_table.to_markdown())

## Phân tích cơ cấu sản phẩm — Tỷ trọng theo phân khúc

**Mục tiêu:**  
Xác định tỷ trọng của từng phân khúc sản phẩm trong danh mục hiện tại.

**Ý nghĩa:**  
Giúp hiểu cấu trúc danh mục sản phẩm của doanh nghiệp, từ đó đánh giá mức độ tập trung hoặc đa dạng hóa sản phẩm theo các phân khúc khác nhau.

**Phương pháp:**  
Thống kê số lượng sản phẩm theo từng phân khúc (segment), sau đó trực quan hóa bằng biểu đồ tròn để thể hiện tỷ trọng của từng nhóm.

### Nhận định chính

- Một số phân khúc chiếm tỷ trọng lớn trong danh mục sản phẩm  
- Sự phân bổ không đồng đều cho thấy doanh nghiệp có thể tập trung vào một số phân khúc nhất định  
- Các phân khúc nhỏ hơn có thể là thị trường ngách hoặc chưa được khai thác nhiều  
- Cơ cấu sản phẩm phản ánh chiến lược phát triển danh mục của doanh nghiệp
- Doanh nghiệp có thể xem xét mở rộng các phân khúc có tiềm năng hoặc tối ưu lại danh mục sản phẩm để cân bằng giữa các nhóm

In [ ]:

# 1. Thống kê số lượng sản phẩm theo từng Segment
segment_counts = df_products['segment'].value_counts()

# 2. Cấu hình biểu đồ
plt.figure(figsize=(10, 8))
colors = sns.color_palette('pastel')[0:len(segment_counts)] # Bộ màu pastel cho dịu mắt

# Tạo hiệu ứng "nổ" nhẹ cho miếng bánh lớn nhất
explode = [0.1 if i == 0 else 0 for i in range(len(segment_counts))]

# 3. Vẽ Pie Chart
plt.pie(
    segment_counts, 
    labels=segment_counts.index, 
    autopct='%1.1f%%',       # Hiển thị phần trăm có 1 chữ số thập phân
    startangle=140,          # Xoay góc bắt đầu cho đẹp
    colors=colors, 
    explode=explode,         # Tách miếng to nhất ra
    shadow=True              # Đổ bóng cho xịn
)

# 4. Trang trí
plt.title('TỶ TRỌNG CÁC PHÂN KHÚC SẢN PHẨM (SEGMENT)', fontsize=14, fontweight='bold')
plt.axis('equal')  # Đảm bảo hình tròn không bị méo thành hình bầu dục

plt.tight_layout()
plt.show()

## Phân tích tính ổn định doanh thu — Phát hiện sản phẩm có doanh thu bằng 0

**Mục tiêu:**  
Xác định các sản phẩm có nhiều năm không phát sinh doanh thu.

**Ý nghĩa:**  
Giúp phát hiện các sản phẩm có hiệu suất kém, không ổn định hoặc đã ngừng kinh doanh trong một số giai đoạn.  
Đây là dấu hiệu quan trọng để đánh giá vòng đời và mức độ hiệu quả của sản phẩm.

**Phương pháp:**  
Dựa trên bảng doanh thu theo năm, chúng tôi đếm số năm mà mỗi sản phẩm có doanh thu bằng 0.  
Sau đó, sắp xếp các sản phẩm theo số năm không có doanh thu (giảm dần) và kết hợp với tổng doanh thu để xác định các sản phẩm có hiệu suất thấp.

### Nhận định chính

- Một số sản phẩm có nhiều năm không phát sinh doanh thu, cho thấy hiệu suất không ổn định  
- Các sản phẩm này có thể đã bị ngừng kinh doanh, lỗi thời hoặc không còn phù hợp với nhu cầu thị trường  
- Sự tồn tại của các sản phẩm “không hoạt động” có thể làm giảm hiệu quả quản lý danh mục sản phẩm  
- Cần xem xét lại vai trò của các sản phẩm này trong chiến lược kinh doanh
- Doanh nghiệp nên cân nhắc loại bỏ, cải tiến hoặc thay thế các sản phẩm có nhiều năm không có doanh thu để tối ưu danh mục sản phẩm

In [ ]:
# 1. Lấy danh sách các cột là Năm (loại bỏ cột Total_All_Years)
year_columns = [col for col in df_pivot_table.columns if col != 'Total_All_Years']

# 2. Đếm số lượng giá trị bằng 0 trên mỗi dòng (mỗi sản phẩm)
# axis=1 là đếm theo hàng ngang
df_pivot_table['zero_years_count'] = (df_pivot_table[year_columns] == 0).sum(axis=1)

# 3. Sắp xếp theo số năm bằng 0 giảm dần, nếu bằng nhau thì ông nào doanh thu thấp hơn đứng trước
df_top_zero = df_pivot_table.sort_values(
    by=['zero_years_count', 'Total_All_Years'], 
    ascending=[False, True]
).head(6)

# 4. In kết quả
print("--- 3 SẢN PHẨM CÓ DOANH THU BẰNG 0 NHIỀU NĂM NHẤT ---")
print(df_top_zero[year_columns + ['zero_years_count', 'Total_All_Years']].to_markdown())

# Xóa cột đếm tạm thời để không làm bẩn bảng chính
df_pivot_table.drop(columns=['zero_years_count'], inplace=True)

## Phân tích khách hàng — Cơ cấu độ tuổi theo khu vực

**Mục tiêu:**  
Phân tích phân bố độ tuổi khách hàng theo từng thành phố có lượng đơn hàng cao nhất.

**Ý nghĩa:**  
Giúp hiểu rõ đặc điểm nhân khẩu học của khách hàng tại các khu vực trọng điểm, từ đó hỗ trợ doanh nghiệp xây dựng chiến lược marketing và cá nhân hóa sản phẩm phù hợp với từng nhóm khách hàng.

**Phương pháp:**  
Dữ liệu đơn hàng được kết hợp với thông tin khách hàng để bổ sung các thuộc tính như thành phố và nhóm tuổi.  
Sau đó, lựa chọn top 5 thành phố có số lượng đơn hàng cao nhất và tiến hành phân tích cơ cấu độ tuổi khách hàng tại các khu vực này.  
Kết quả được trực quan hóa bằng biểu đồ cột chồng để thể hiện tỷ trọng từng nhóm tuổi.

### Nhận định chính

- Nhóm khách hàng từ 25–34 tuổi chiếm tỷ trọng cao nhất trong tổng số đơn hàng  
- Các thành phố lớn có sự phân bố độ tuổi khách hàng đa dạng hơn so với các khu vực khác  
- Một số nhóm tuổi chiếm ưu thế rõ rệt tại từng khu vực, cho thấy sự khác biệt về hành vi tiêu dùng  
- Cơ cấu độ tuổi khách hàng phản ánh đặc điểm thị trường và nhu cầu tiêu dùng tại từng thành phố
- Doanh nghiệp có thể tập trung chiến lược marketing vào nhóm khách hàng 25–34 tuổi, đồng thời điều chỉnh nội dung và sản phẩm phù hợp với từng khu vực để tối ưu hiệu quả kinh doanh

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Nối 2 bảng để lấy cả City và Age_Group
df_merged = df_orders.merge(df_customers[['customer_id', 'city', 'age_group']], on='customer_id', how='left')

# 2. Tìm Top 5 thành phố có lượng đơn hàng nhiều nhất (để lọc dữ liệu)
top_5_cities = df_merged['city'].value_counts().head(5).index.tolist()

# 3. Lọc dữ liệu chỉ lấy Top 5 thành phố này
df_top5 = df_merged[df_merged['city'].isin(top_5_cities)].copy()

# 4. Gom nhóm theo City và Age_Group để đếm số đơn hàng
df_pivot = df_top5.groupby(['city', 'age_group']).size().unstack(fill_value=0)

# 5. Sắp xếp lại các thành phố theo tổng lượng đơn hàng giảm dần để biểu đồ đẹp hơn
df_pivot['total'] = df_pivot.sum(axis=1)
df_pivot = df_pivot.sort_values(by='total', ascending=False).drop(columns='total')

# 6. Vẽ biểu đồ cột chồng (Stacked Bar Chart)
plt.figure(figsize=(14, 8))
sns.set_theme(style="whitegrid")

# Vẽ bằng pandas plot cho nhanh và chuẩn với dạng pivot
ax = df_pivot.plot(kind='bar', stacked=True, figsize=(12, 7), 
                   color=sns.color_palette("viridis", len(df_pivot.columns)))

# 7. Trang trí
plt.title('TOP 5 THÀNH PHỐ VÀ CƠ CẤU ĐỘ TUỔI KHÁCH HÀNG ORDER', fontsize=15, fontweight='bold', pad=20)
plt.xlabel('Thành phố (Region)', fontsize=12)
plt.ylabel('Số lượng đơn hàng', fontsize=12)
plt.xticks(rotation=0) # Để tên thành phố nằm ngang cho dễ đọc

# Hiển thị chú thích nhóm tuổi
plt.legend(title='Nhóm tuổi', bbox_to_anchor=(1.05, 1), loc='upper left')

# Thêm con số tổng lên đầu mỗi cột cho dễ nhìn
totals = df_pivot.sum(axis=1).values
for i, total in enumerate(totals):
    plt.text(i, total + 0.5, f'{int(total)}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# In bảng pivot ra để ông kiểm tra số liệu
print("--- CHI TIẾT ĐƠN HÀNG THEO VÙNG VÀ ĐỘ TUỔI ---")
print(df_pivot.to_markdown())
print("---------------------------------------------------")
print("Tỷ trọng ở lớp tuổi 25-34 chiếm tỷ trọng cao nhất")


## Phân tích vận hành — Hiệu suất giao hàng theo thời gian

**Mục tiêu:**  
Đánh giá hiệu suất giao hàng của doanh nghiệp theo từng năm.

**Ý nghĩa:**  
Giúp hiểu rõ mức độ hiệu quả trong hoạt động logistics, từ đó xác định xu hướng cải thiện hoặc suy giảm chất lượng dịch vụ giao hàng.

**Phương pháp:**  
Thời gian giao hàng được tính bằng khoảng cách giữa ngày giao hàng và ngày gửi hàng.  
Sau đó, dữ liệu được tổng hợp theo từng năm để tính các chỉ số quan trọng như:
- Tổng số đơn hàng  
- Thời gian giao trung bình  
- Thời gian giao nhanh nhất và chậm nhất  
- Trung vị thời gian giao  

Kết quả được trình bày dưới dạng bảng thống kê để so sánh qua các năm.

### Nhận định chính

- Thời gian giao hàng trung bình phản ánh hiệu suất vận hành tổng thể của doanh nghiệp  
- Sự thay đổi qua các năm cho thấy mức độ cải thiện hoặc suy giảm trong quy trình giao hàng  
- Khoảng cách giữa giá trị nhanh nhất và chậm nhất cho thấy mức độ biến động trong dịch vụ giao hàng  
- Trung vị giúp phản ánh chính xác hơn trải nghiệm phổ biến của khách hàng

In [ ]:
import pandas as pd
print("-----THỐNG KÊ DỮ LIỆU GIAO HÀNG THEO TỪNG NĂM--------------")
# 1. Chuẩn bị dữ liệu
df_shipments_1 = df_shipments.copy()
df_shipments_1['delivery_date'] = pd.to_datetime(df_shipments_1['delivery_date'])
df_shipments_1['ship_date'] = pd.to_datetime(df_shipments_1['ship_date'])

# 2. Tính duration và trích xuất Year
df_shipments_1['duration'] = (df_shipments_1['delivery_date'] - df_shipments_1['ship_date']).dt.days
df_shipments_1['year'] = df_shipments_1['ship_date'].dt.year

# 3. Lọc rác
df_shipments_1 = df_shipments_1[df_shipments_1['duration'] >= 0].dropna(subset=['duration'])

# 4. Tạo bảng thống kê tổng hợp (Pivot Table / Groupby)
summary_table = df_shipments_1.groupby('year')['duration'].agg([
    'count',      # Tổng số đơn hàng
    'mean',       # Số ngày giao trung bình
    'min',        # Giao nhanh kỷ lục (số ngày)
    'max',        # Giao chậm kỷ lục (số ngày)
    'median'      # Trung vị (mức phổ biến nhất)
]).reset_index()

# Làm tròn số cho đẹp
summary_table['mean'] = summary_table['mean'].round(2)

# Đổi tên cột cho dễ đọc
summary_table.columns = ['Năm', 'Tổng số đơn', 'Trung bình (ngày)', 'Nhanh nhất', 'Chậm nhất', 'Trung vị']

print(summary_table)

## Phân tích tồn kho và hiệu suất bán hàng

**Mục tiêu:**  
So sánh giữa số lượng sản phẩm bán ra và mức tồn kho để đánh giá hiệu quả quản lý hàng hóa.

**Ý nghĩa:**  
Giúp xác định các sản phẩm bán chạy và các sản phẩm tồn kho nhiều, từ đó phát hiện sự mất cân đối giữa cung và cầu.  
Đây là cơ sở quan trọng để tối ưu quản lý tồn kho và giảm chi phí lưu trữ.

**Phương pháp:**  
Dữ liệu được tổng hợp theo từng sản phẩm để tính:
- Tổng số lượng bán ra (units_sold)  
- Mức tồn kho trung bình (stock_on_hand)  

Sau đó, lựa chọn top 8 sản phẩm bán chạy nhất và top 8 sản phẩm có tồn kho nhiều nhất.  
Kết quả được trực quan hóa bằng hai biểu đồ cột để so sánh trực tiếp.

### Nhận định chính

- Một số sản phẩm có lượng bán cao, cho thấy nhu cầu thị trường lớn  
- Một số sản phẩm có tồn kho cao nhưng chưa chắc bán tốt, có thể đang dư thừa hàng hóa  
- Sự chênh lệch giữa bán ra và tồn kho phản ánh mức độ hiệu quả trong quản lý cung cầu  
- Các sản phẩm tồn kho cao có thể gây phát sinh chi phí lưu trữ và rủi ro lỗi thời

In [ ]:
# Tính tổng lượng bán (units_sold) và trung bình tồn kho (stock_on_hand) theo product_id
agg_df = df_inventory.groupby('product_id').agg({
    'units_sold': 'sum',
    'stock_on_hand': 'mean'
}).reset_index()

# 3. Merge với bảng productid để lấy product_name cho dễ đọc
final_df = pd.merge(agg_df, df_products[['product_id', 'product_name']], on='product_id')

# 4. Lấy Top 8
top_8_sold = final_df.sort_values('units_sold', ascending=False).head(8)
top_8_stock = final_df.sort_values('stock_on_hand', ascending=False).head(8)

# 5. Vẽ Chart
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Chart 1: Top 8 Bán ra
sns.barplot(ax=axes[0], data=top_8_sold, x='units_sold', y='product_name', palette='viridis')
axes[0].set_title('Top 8 Sản phẩm bán chạy nhất (Tổng số lượng)', fontsize=14)
axes[0].set_xlabel('Số lượng bán ra')

# Chart 2: Top 8 Tồn kho
sns.barplot(ax=axes[1], data=top_8_stock, x='stock_on_hand', y='product_name', palette='magma')
axes[1].set_title('Top 8 Sản phẩm tồn kho nhiều nhất (Trung bình)', fontsize=14)
axes[1].set_xlabel('Số lượng tồn kho trung bình')

plt.tight_layout()
plt.show()

## Trích xuất biểu đồ từ Notebook

**Mục tiêu:**  
Tự động trích xuất các biểu đồ đã tạo trong notebook để phục vụ cho việc xây dựng báo cáo.

**Ý nghĩa:**  
Giúp tiết kiệm thời gian và đảm bảo tính nhất quán khi đưa các biểu đồ vào báo cáo kỹ thuật.  
Đồng thời hỗ trợ việc trình bày kết quả phân tích một cách chuyên nghiệp và trực quan.

**Phương pháp:**  
Duyệt qua các cell trong notebook, xác định các output chứa hình ảnh (định dạng PNG hoặc JPEG),  
sau đó giải mã dữ liệu và lưu thành các file ảnh riêng biệt trong thư mục chỉ định.

In [ ]:
import nbformat
import base64
import os

# 1. Cấu hình tên file và thư mục lưu ảnh
notebook_path = 'descriptive.ipynb'
output_dir = 'code/eda/extracted_images'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Đọc file notebook
with open(notebook_path, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

image_count = 0

# 3. Duyệt qua từng cell, tìm output là hình ảnh
for i, cell in enumerate(nb.cells):
    if cell.cell_type == 'code':
        for j, output in enumerate(cell.get('outputs', [])):
            # Kiểm tra nếu output chứa dữ liệu hình ảnh (thường là png hoặc jpeg)
            if 'data' in output and ('image/png' in output['data'] or 'image/jpeg' in output['data']):
                
                # Xác định định dạng ảnh
                fmt = 'png' if 'image/png' in output['data'] else 'jpeg'
                image_data = output['data'][f'image/{fmt}']
                
                # Giải mã base64 và lưu file
                file_name = f'chart_cell{i}_{j}.{fmt}'
                file_path = os.path.join(output_dir, file_name)
                
                with open(file_path, 'wb') as img_f:
                    img_f.write(base64.b64decode(image_data))
                
                print(f"Đã lưu: {file_path}")
                image_count += 1

print(f"\n--- Xong! Đã hốt được {image_count} cái ảnh bỏ vào thư mục '{output_dir}' ---")

## RFM statistics (Cái này của diagnostic và descriptive)
### Target
* Xem lượt mua gần nhất của khách hàng là bao lâu (Recency)
* Dùng thuật toán K-Mean Clustering để phân ra 3 vùng (0 - 1 tháng, 1 - 6 tháng, 2 - 1 năm)
* Xem các khách hàng ở phân khúc này thường độ tuổi là bao nhiêu

In [ ]:
df_rfm = pd.merge(df_orders,df_customers, left_on = "customer_id", right_on = "customer_id", how = "left")


In [ ]:
df_rfm.columns

In [ ]:
df_rfm = df_rfm.drop(columns=['zip_x', 'city', 'signup_date', 'gender', 'age_group',
       'acquisition_channel','zip_y'])

In [ ]:
df_rfm

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

In [ ]:
df_rfm['order_date'] = pd.to_datetime(df_rfm['order_date'])

# Giả định ngày hiện tại để tính Recency là ngày sau đơn hàng cuối cùng 1 ngày
current_date = df_rfm['order_date'].max() + dt.timedelta(days=1)

# Gom nhóm theo customer_id để tính RFM
# Chú ý: Vì bảng bạn không có cột 'Revenue/Amount', tôi sẽ coi mỗi đơn hàng có giá trị mặc định là 1 
# hoặc bạn hãy thay 'order_id' bằng cột số tiền nếu có.
rfm = df_rfm.groupby('customer_id').agg({
    'order_date': lambda x: (current_date - x.max()).days, # Recency
    'order_id': 'count',                                   # Frequency
}).reset_index()

rfm.columns = ['customer_id', 'Recency', 'Frequency']

In [ ]:
rfm

In [ ]:
rfm.isna().sum()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np
df_rfm
# 1. Log Transformation để giảm độ lệch (Skewness)
rfm_log = np.log(rfm[['Recency', 'Frequency']] + 1)

# 2. Chuẩn hóa
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# 3. Phân thành 3 cụm (như bạn muốn: 1 tháng, 6 tháng, 1 năm)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [ ]:
plt.figure(figsize=(10, 6))

# Vẽ biểu đồ phân phối
sns.histplot(rfm['Recency'], kde=True, color='blue', bins=30)

plt.title('Biểu đồ phân phối của Recency (Check Skewness)')
plt.xlabel('Số ngày kể từ lần mua cuối (Recency)')
plt.ylabel('Số lượng khách hàng')
plt.show()

# Tính toán con số cụ thể
print(f"Độ lệch (Skewness) của Recency là: {rfm['Recency'].skew()}")

In [ ]:
data_positive = rfm['Recency'] + 1

# 2. Thực hiện Box-Cox
# Trả về 2 giá trị: dữ liệu đã biến đổi và giá trị lambda tối ưu
data_boxcox, lam = stats.boxcox(data_positive)

rfm['Recency_boxcox'] = data_boxcox

print(f"Lambda tối ưu tìm được: {lam}")
print(f"Độ lệch (Skewness) sau Box-Cox: {rfm['Recency_boxcox'].skew()}")
# 1. Chuẩn bị dữ liệu (Gốc, Log, Box-Cox)
res_original = rfm['Recency'] + 1
res_log = np.log1p(rfm['Recency'])
res_boxcox, lam = stats.boxcox(res_original)

# 2. Thiết kế khung biểu đồ
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Biểu đồ 1: Gốc (Skewness ~0.66) ---
sns.histplot(res_original, kde=True, ax=axes[0], color='red')
axes[0].set_title(f"Gốc (Skew: {res_original.skew():.3f})")
axes[0].set_xlabel("Số ngày (Recency)")

# --- Biểu đồ 2: Log Transformation ---
sns.histplot(res_log, kde=True, ax=axes[1], color='blue')
axes[1].set_title(f"Sau Log (Skew: {res_log.skew():.3f})")
axes[1].set_xlabel("Log Value")

# --- Biểu đồ 3: Box-Cox Transformation ---
sns.histplot(res_boxcox, kde=True, ax=axes[2], color='green')
axes[2].set_title(f"Sau Box-Cox (Skew: {pd.Series(res_boxcox).skew():.3f})")
axes[2].set_xlabel(f"Box-Cox Value (Lambda: {lam:.2f})")

plt.tight_layout()
plt.show()

In [ ]:
# Left Join kết quả Cluster vào bảng gốc
df_final = pd.merge(df_rfm, rfm[['customer_id', 'Cluster']], on='customer_id', how='left')

# Xóa nhiều cột thừa cùng lúc để làm sạch bảng cuối cùng
# Ví dụ: xóa device_type, order_source và payment_method nếu không dùng tới
cols_to_drop = ['device_type', 'order_source', 'payment_method', 'order_status']
df_final.drop(columns=cols_to_drop, inplace=True)

print(df_final.head())
print(df_final['Cluster'].nunique())

In [ ]:
df_final['Cluster'].value_counts()

In [ ]:
print(df_shipments.columns)

In [ ]:
df_rfm_ship = pd.merge(df_final, df_shipments, left_on = "order_id", right_on = "order_id", how = "left")
df_rfm_ship = df_rfm_ship.drop(columns=['shipping_fee'])

In [ ]:
df_promotions

In [ ]:
df_rfm_ship['ship_date'].isna().sum()

In [67]:
df_special = pd.merge(df_orders, df_order_items, left_on = "order_id", right_on = "order_id", how = "left")
df_special['order_date'] = pd.to_datetime(df_special['order_date'])
df_special = df_special.drop(columns=['order_status', 'order_source', 'device_type', 'payment_method'])

In [68]:
df_special

,order_id,order_date,customer_id,zip,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2012-07-04,58578,1109,2400,7,"1,138",0,NaN,NaN
1,2,2012-07-04,58621,1330,609,7,"10,166",0,NaN,NaN
2,3,2012-07-04,58811,1473,396,3,"11,220",0,NaN,NaN
3,4,2012-07-04,59453,2360,635,5,"10,639",0,NaN,NaN
4,6,2012-07-06,57821,2886,1935,1,"1,598",0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
714664,834372,2022-12-31,19490,33907,690,8,"4,474",0,NaN,NaN
714665,834377,2022-12-31,73046,37091,1995,7,"5,251",0,NaN,NaN
714666,834387,2022-12-31,107723,80516,2331,8,"7,389",0,NaN,NaN
714667,834392,2022-12-31,139431,93510,1115,5,"4,767",0,NaN,NaN


In [70]:
# Chuyển đổi sang datetime nếu chưa làm
df_special['order_date'] = pd.to_datetime(df_special['order_date'])

# Lọc tháng 8 của năm 2013
thang_8 = df_special[(df_special['order_date'].dt.month == 8) & 
                     (df_special['order_date'].dt.year == 2013)]

# Hoặc lọc tháng 8 của TẤT CẢ các năm
thang_8

,order_id,order_date,customer_id,zip,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
89644,101799,2013-08-01,59202,2025,2354,6,"4,442",0,NaN,NaN
89645,101800,2013-08-01,59303,2072,2054,7,"6,823",0,NaN,NaN
89653,101808,2013-08-01,49097,5261,2082,8,"4,038",0,NaN,NaN
89657,101813,2013-08-01,57014,6450,1916,3,"3,464",0,NaN,NaN
89659,101815,2013-08-01,57373,6776,604,2,"6,331",0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
100240,114034,2013-08-31,142132,93436,606,4,"3,496","1,399",PROMO-0003,NaN
100259,114052,2013-08-31,135253,95377,498,7,"16,166","11,316",PROMO-0003,NaN
100266,114061,2013-08-31,136995,96049,676,5,"8,998","4,499",PROMO-0003,NaN
100269,114065,2013-08-31,150486,97386,1182,1,"2,067",207,PROMO-0003,NaN


In [73]:
thang_8['promo_id'].notnull().sum()

np.int64(4444)

In [74]:
# Chuyển đổi sang datetime nếu chưa làm
df_special['order_date'] = pd.to_datetime(df_special['order_date'])

# Lọc tháng 5 của năm 2018
thang_5_2018 = df_special[(df_special['order_date'].dt.month == 5) & 
                          (df_special['order_date'].dt.year == 2018)]

In [75]:
thang_5_2018['promo_id'].notnull().sum()


np.int64(0)